In [1]:
from datasets import load_dataset
dataset_train = load_dataset('glue', 'mrpc', split='train')
dataset_test = load_dataset('glue', 'mrpc', split='test')
dataset_val = load_dataset('glue', 'mrpc', split='validation')

/home/ubuntu/myenv/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Found cached dataset glue (/home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
Found cached dataset glue (/home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
Found cached dataset glue (/home/ubuntu/.cache/huggingface/datasets/glue/mrpc/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased')
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPretraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at b

In [3]:
def encode(examples):
    return tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding='max_length')

dataset_train = dataset_train.map(encode, batched=True)
#Rename label to labels
dataset_train = dataset_train.map(lambda rows: {'labels': rows['label']}, batched=True)

In [4]:
import torch
dataset_train.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
dataloader = torch.utils.data.DataLoader(dataset_train, batch_size=32)

In [ ]:
#Train model
from tqdm import tqdm
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device is {device}')
model.train().to(device)
optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-5)
for epoch in range(3):
    for i, batch in enumerate(tqdm(dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs[0]
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if i % 10 == 0:
            print(f"loss: {loss}")

device is cpu


  1%|▊                                                                                      | 1/115 [01:10<2:13:01, 70.01s/it]

loss: 0.7743574976921082


 10%|████████▏                                                                             | 11/115 [12:49<2:01:21, 70.02s/it]

loss: 0.7205988168716431


...

 36%|██████████████████████████████▋                                                       | 41/115 [47:25<1:25:37, 69.42s/it]

loss: 0.42815396189689636


 44%|██████████████████████████████████████▏                                               | 51/115 [58:59<1:13:58, 69.34s/it]

loss: 0.23625516891479492


 52%|███████████████████████████████████████████▊                                        | 60/115 [1:09:23<1:03:34, 69.36s/it]